# 02 — Candle Primitives (Real Data, All Timeframes)

## Goal
Compute the per-candle features that every downstream step depends on — for **all four timeframes**.

## Equations

**Range** — the full price span of a candle:
$$\text{range} = \text{high} - \text{low}$$

**Body** — the absolute distance between open and close:
$$\text{body} = |\text{close} - \text{open}|$$

**Body Ratio** — fraction of the range that is real move vs shadow:
$$\text{body\_ratio} = \frac{\text{body}}{\text{range}}$$

- $\text{body\_ratio} \approx 1.0$ → strong directional candle (almost no wicks)
- $\text{body\_ratio} \approx 0.0$ → doji or wide-shadow candle

## Classification Rules

| Flag | Condition | Meaning |
|------|-----------|---------|
| `is_base` | $\text{body\_ratio} \leq 0.50$ | Indecisive / consolidation candle |
| `is_doji` | $\text{body\_ratio} \leq 0.10$ | Near-zero body (perfect indecision) |
| `is_bullish` | $\text{close} > \text{open}$ | Buyers won the session |

## Visualization
Each timeframe gets its own two-panel chart:
- **Top panel** — candlesticks color-coded by type (teal=bullish, red=bearish, grey=base, gold=doji)
- **Bottom panel** — `body_ratio` bar, dashed threshold at 0.50


## 1. Setup & load data

In [5]:
import sys
sys.path.append("..")

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from utils.data_loader import load_all_timeframes
from utils.models import CandlePrimitives
from utils.config import (
    BASE_BODY_RATIO_MAX, DOJI_BODY_RATIO_MAX,
    COLOR_BULL, COLOR_BEAR, COLOR_BASE, COLOR_DOJI,
    CHART_BG as BG, CHART_GRID as GRID,
)

pio.renderers.default = "notebook_connected"

SYMBOL = "USDJPY=X"   # ← change to any downloaded symbol

# ── load all 4 TFs ───────────────────────────────────────────────────────────
data = load_all_timeframes(SYMBOL, align=True)
print(f"Loaded {SYMBOL} — {list(data.keys())}")
for tf, df in data.items():
    print(f"  {tf}: {len(df)} rows")


Loaded USDJPY=X — ['1wk', '1d', '4h', '1h']
  1wk: 104 rows
  1d: 517 rows
  4h: 3175 rows
  1h: 12262 rows


## 2. Compute primitives for every timeframe

Uses `CandlePrimitives.enrich_dataframe()` from `utils/models.py` — the same logic the production pipeline uses.

Columns added:

| Column | Formula |
|--------|---------|
| `candle_range` | `high − low` |
| `body_size` | `abs(close − open)` |
| `body_to_range_ratio` | `body_size / candle_range` |
| `is_base` | `body_to_range_ratio <= 0.50` |
| `is_doji` | `body_to_range_ratio <= 0.10` |
| `is_bullish` | `close > open` |
| `true_range` | `max(range, \|high−prev_close\|, \|low−prev_close\|)` |

Zero-range candles (market halt / weekend FX gaps) are assigned `body_to_range_ratio = 0` so they are treated as doji.


In [6]:
data = {tf: CandlePrimitives.enrich_dataframe(df) for tf, df in data.items()}

# ── quick stats across all TFs ───────────────────────────────────────────────
print(f"{'TF':<6} {'Rows':>6}  {'Base%':>7}  {'Doji%':>7}  {'Bull%':>7}")
print("-" * 42)
for tf, df in data.items():
    n = len(df)
    print(f"{tf:<6} {n:>6}  "
          f"{df['is_base'].mean()*100:>6.1f}%  "
          f"{df['is_doji'].mean()*100:>6.1f}%  "
          f"{df['is_bullish'].mean()*100:>6.1f}%")


TF       Rows    Base%    Doji%    Bull%
------------------------------------------
1wk       104    60.6%     8.7%    55.8%
1d        517   100.0%    98.3%    15.5%
4h       3175    58.4%    12.4%    51.7%
1h      12262    58.8%    11.3%    50.8%


## 3. Inspect — daily sample

In [7]:
data["1d"][["open","high","low","close",
            "candle_range","body_size","body_to_range_ratio",
            "is_base","is_doji","is_bullish"]].head(10)


,open,high,low,close,candle_range,body_size,body_to_range_ratio,is_base,is_doji,is_bullish
Date,,,,,,,,,,
2024-06-04 23:00:00+00:00,155.123993,156.473007,155.110992,155.123993,1.362015,0.0,0.0,True,True,False
2024-06-05 23:00:00+00:00,155.929001,156.395996,155.406998,155.929001,0.988998,0.0,0.0,True,True,False
2024-06-06 23:00:00+00:00,155.697006,157.067001,155.141998,155.697006,1.925003,0.0,0.0,True,True,False
2024-06-09 23:00:00+00:00,156.832993,157.186005,156.723007,156.832993,0.462997,0.0,0.0,True,True,False
2024-06-10 23:00:00+00:00,157.039993,157.388000,156.811005,157.039993,0.576996,0.0,0.0,True,True,False
2024-06-11 23:00:00+00:00,157.104996,157.365005,155.729004,157.104996,1.636002,0.0,0.0,True,True,False
2024-06-12 23:00:00+00:00,156.800003,157.300003,156.576996,156.800003,0.723007,0.0,0.0,True,True,False
2024-06-13 23:00:00+00:00,157.128006,158.246994,156.899994,157.128006,1.347000,0.0,0.0,True,True,False
2024-06-16 23:00:00+00:00,157.546005,157.951996,157.169998,157.546005,0.781998,0.0,0.0,True,True,False


## 4. Candlestick charts — color coded by type (all 4 TFs)

One chart per timeframe, last 120 candles. Color scheme:

| Color | Candle type |
|-------|------------|
| Teal  | Bullish (non-base) |
| Red   | Bearish (non-base) |
| Grey  | Base (body_ratio <= 0.50) |
| Gold  | Doji (body_ratio <= 0.10) — overrides base |

Bottom panel: `body_ratio` bar using the same color as the candle above it.


In [8]:
def candle_color(row):
    if row["is_doji"]: return COLOR_DOJI
    if row["is_base"]: return COLOR_BASE
    return COLOR_BULL if row["is_bullish"] else COLOR_BEAR

def plot_primitives(df, tf: str, n_candles: int = 120):
    d      = df.tail(n_candles)
    colors = d.apply(candle_color, axis=1)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.70, 0.30],
        vertical_spacing=0.03,
        subplot_titles=[f"{SYMBOL} {tf} — Candle Types", "Body-to-Range Ratio"],
    )

    fig.add_trace(go.Candlestick(
        x=d.index,
        open=d["open"], high=d["high"],
        low=d["low"],   close=d["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=d.index, y=d["body_to_range_ratio"],
        marker_color=colors,
        name="Body Ratio", showlegend=False,
    ), row=2, col=1)

    fig.add_hline(
        y=BASE_BODY_RATIO_MAX,
        line_dash="dot", line_color="#888", line_width=1,
        annotation_text=f"base <= {BASE_BODY_RATIO_MAX}",
        annotation_font_color="#888",
        annotation_position="bottom right",
        row=2, col=1,
    )

    fig.update_layout(
        height=520,
        plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.06),
        bargap=0.15,
    )
    for ax in ["xaxis", "xaxis2", "yaxis", "yaxis2"]:
        fig.update_layout(**{ax: dict(gridcolor=GRID, showgrid=True, zeroline=False)})
    fig.update_yaxes(title_text="Price",            row=1, col=1)
    fig.update_yaxes(title_text="Body/Range Ratio", row=2, col=1, range=[0, 1.05])
    fig.show()

for tf in ["1wk", "1d", "4h", "1h"]:
    plot_primitives(data[tf], tf)


## 5. What's next

| Notebook | Topic |
|----------|-------|
| `03_atr.ipynb` | Compute ATR(14) on each timeframe — the volatility ruler for all zone width checks |
| `04_base_detection.ipynb` | Cluster base candles into zones using ATR as the width guard |

The enriched `data` dict (all 4 TFs with primitives) is the input to NB03.
